In [11]:
%autosave 60

Autosaving every 60 seconds


# RAG: влияние промпта, эмбеддинга и LLM

Сравнение качества RAG при изменении **промпта**, **модели эмбеддингов** и **LLM** на едином бенчмарке (PDF Ростехнадзор, 25 сэмплов). Метрики: retrieval (recall@k, MRR) и generation (ROUGE, BLEU, BERTScore).

## Настройка окружения

Загружаем `.env`, подключаем модели из `config`, инициализируем два LLM (baseline и v2), два варианта эмбеддингов (bge-m3 и google/gemini-embedding-001 через Open Router). Данные — PDF Ростехнадзор (`data_v2.pdf`) и бенчмарк из артефактов.

In [17]:
import os
import sys
import json
import requests
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    from dotenv import load_dotenv
    load_dotenv(env_path)

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

from config import model_llm, model_llm_v2, model_embedding, model_embedding_v2

PATH_PDF = PROJECT_ROOT / "data" / "text" / "data_v2.pdf"

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model=model_llm,
    default_headers={"X-Title": "RAG Improvements Notebook"},
)
llm_v2 = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model=model_llm_v2,
    default_headers={"X-Title": "RAG Improvements Notebook"},
)
embeddings = HuggingFaceEmbeddings(model_name=model_embedding)


class OpenRouterEmbeddings(Embeddings):
    """Эмбеддинги через Open Router API (например google/gemini-embedding-001)."""

    def __init__(self, api_key: str, model: str, batch_size: int = 100):
        self.api_key = api_key
        self.model = model
        self.batch_size = batch_size
        self.url = "https://openrouter.ai/api/v1/embeddings"

    def _embed(self, texts: list[str]) -> list[list[float]]:
        out = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            payload = {
                "model": self.model,
                "input": batch if len(batch) > 1 else batch[0],
                "encoding_format": "float",
            }
            resp = requests.post(
                self.url,
                headers={
                    "Authorization": f"Bearer {self.api_key}",
                    "Content-Type": "application/json",
                    "X-Title": "RAG Improvements Notebook",
                },
                json=payload,
                timeout=60,
            )
            resp.raise_for_status()
            data = resp.json()
            for item in data["data"]:
                out.append(item["embedding"])
        return out

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts)

    def embed_query(self, text: str) -> list[float]:
        return self._embed([text])[0]


embeddings_v2 = OpenRouterEmbeddings(
    api_key=os.environ.get("OPENROUTER_API_KEY", ""),
    model=model_embedding_v2,
)

print("Окружение готово. LLM:", model_llm, "| LLM v2:", model_llm_v2)
print("Эмбеддинги:", model_embedding, "| v2:", model_embedding_v2)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1617.66it/s, Materializing param=pooler.dense.weight]                               


Окружение готово. LLM: google/gemini-3.1-flash-lite-preview | LLM v2: google/gemini-3.1-pro-preview
Эмбеддинги: baai/bge-m3 | v2: google/gemini-embedding-001


In [13]:
# Загрузка PDF и baseline-чанкинг (как в ноутбуке 3)
loader_pdf = PyPDFLoader(str(PATH_PDF))
docs_pdf_raw = loader_pdf.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits_pdf = text_splitter.split_documents(docs_pdf_raw)
for i, doc in enumerate(splits_pdf):
    doc.metadata["chunk_idx"] = i

artifacts_dir = PROJECT_ROOT / "notebooks_rag" / "artifacts"
with open(artifacts_dir / "samples_pdf.json", "r", encoding="utf-8") as f:
    samples_pdf = json.load(f)

for s in samples_pdf:
    idx = s["chunk_idx"]
    s["reference_chunk_text"] = splits_pdf[idx].page_content

print("Чанков (baseline):", len(splits_pdf), "| Сэмплов в бенчмарке:", len(samples_pdf))

Чанков (baseline): 1300 | Сэмплов в бенчмарке: 25


In [14]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Baseline-промпт (как в ноутбуке 3)
system_prompt_baseline = (
    "Ты — полезный ИИ-ассистент. Используй следующий контекст для ответа на вопрос пользователя. "
    "Если ответа в контексте нет, так и скажи.\n\nКонтекст: {context}"
)
prompt_baseline = ChatPromptTemplate.from_messages([
    ("system", system_prompt_baseline),
    ("human", "{question}"),
])

# Улучшенный промпт: роль, явные инструкции, краткость
system_prompt_improved = (
    "Ты — эксперт по нормативным документам. Отвечай на вопрос **только** на основе приведённого контекста. "
    "Не придумывай факты. Если в контексте нет ответа — чётко напиши «В контексте нет ответа на этот вопрос». "
    "Будь кратким и по делу; при необходимости оформи ответ короткими пунктами.\n\nКонтекст:\n{context}"
)
prompt_improved = ChatPromptTemplate.from_messages([
    ("system", system_prompt_improved),
    ("human", "{question}"),
])

## 1) Baseline — исходный замер

Векторный store на baseline-эмбеддингах (bge-m3), retriever k=10, baseline-промпт и baseline-LLM. С этим конфигом будем сравнивать все последующие варианты.

In [5]:
vectorstore_baseline = Chroma.from_documents(
    documents=splits_pdf,
    embedding=embeddings,
    collection_name="rag_improvements_baseline",
)
retriever_baseline = vectorstore_baseline.as_retriever(search_kwargs={"k": 10})
rag_chain_baseline = (
    {"context": retriever_baseline | format_docs, "question": RunnablePassthrough()}
    | prompt_baseline
    | llm
    | StrOutputParser()
)
print("Baseline RAG собран (retriever k=10).")

Baseline RAG собран (retriever k=10).


In [6]:
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import score as bert_score_fun

K_LIST = [1, 3, 5, 10]
retrieval_rows = []
for i, s in enumerate(samples_pdf):
    retrieved = retriever_baseline.invoke(s["question"])
    retrieved_indices = [d.metadata.get("chunk_idx", -1) for d in retrieved]
    rel_idx = s["chunk_idx"]
    rank = 0
    for pos, idx in enumerate(retrieved_indices, 1):
        if idx == rel_idx:
            rank = pos
            break
    recall_at_k = {f"recall@{k}": (1.0 if rel_idx in retrieved_indices[:k] else 0.0) for k in K_LIST}
    mrr = 1.0 / rank if rank else 0.0
    retrieval_rows.append({"sample_id": i, **recall_at_k, "mrr": mrr})

df_ret_baseline = pd.DataFrame(retrieval_rows)
means_ret_baseline = df_ret_baseline.drop(columns=["sample_id"]).mean()
print("Retrieval (baseline), средние:")
print(means_ret_baseline.to_string())
df_ret_baseline

Retrieval (baseline), средние:
recall@1     0.640000
recall@3     0.880000
recall@5     0.960000
recall@10    0.960000
mrr          0.771333


,sample_id,recall@1,recall@3,recall@5,recall@10,mrr
0,0,0.0,0.0,1.0,1.0,0.200000
1,1,1.0,1.0,1.0,1.0,1.000000
2,2,1.0,1.0,1.0,1.0,1.000000
3,3,0.0,1.0,1.0,1.0,0.500000
4,4,0.0,1.0,1.0,1.0,0.333333
5,5,1.0,1.0,1.0,1.0,1.000000
6,6,1.0,1.0,1.0,1.0,1.000000
7,7,0.0,1.0,1.0,1.0,0.500000
8,8,1.0,1.0,1.0,1.0,1.000000
9,9,1.0,1.0,1.0,1.0,1.000000


In [7]:
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)
answers_baseline = [rag_chain_baseline.invoke(s["question"]) for s in samples_pdf]
refs_pdf = [s["ground_truth"][:500] for s in samples_pdf]
gen_rows = []
for i, (answer, ref) in enumerate(zip(answers_baseline, refs_pdf)):
    r1 = rouge.score(ref, answer)["rouge1"].fmeasure
    rL = rouge.score(ref, answer)["rougeL"].fmeasure
    bleu = sacrebleu.sentence_bleu(answer, [ref], smooth_method="exp").score / 100.0
    gen_rows.append({"sample_id": i, "rouge1_f": r1, "rougeL_f": rL, "bleu": bleu})
df_gen_baseline = pd.DataFrame(gen_rows)
_, _, F1 = bert_score_fun(cands=answers_baseline, refs=refs_pdf, lang="ru", verbose=False)
df_gen_baseline["bertscore_f1"] = F1.numpy()
means_gen_baseline = df_gen_baseline.drop(columns=["sample_id"]).mean()
print("Generation (baseline), средние:")
print(means_gen_baseline.to_string())
df_gen_baseline

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1657.38it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generation (baseline), средние:
rouge1_f        0.205623
rougeL_f        0.199223
bleu            0.108106
bertscore_f1    0.755252


,sample_id,rouge1_f,rougeL_f,bleu,bertscore_f1
0,0,0.173913,0.173913,0.023629,0.670994
1,1,0.500000,0.500000,0.076391,0.742125
2,2,0.000000,0.000000,0.021102,0.793410
3,3,0.000000,0.000000,0.075959,0.698020
4,4,0.000000,0.000000,0.126257,0.796640
5,5,0.000000,0.000000,0.086133,0.733635
6,6,0.000000,0.000000,0.140346,0.785777
7,7,0.000000,0.000000,0.046056,0.725944
8,8,0.000000,0.000000,0.008392,0.723896
9,9,0.000000,0.000000,0.112355,0.744045


**Пример на одном сэмпле (baseline):** вопрос, фрагмент контекста, ответ модели и эталон — для наглядности.

In [8]:
ex_idx = 0
s = samples_pdf[ex_idx]
ctx = retriever_baseline.invoke(s["question"])
ctx_text = format_docs(ctx)
print("Вопрос:", s["question"][:200], "...")
print("\nКонтекст (начало):", ctx_text[:400], "...")
print("\nОтвет модели:", answers_baseline[ex_idx][:300])
print("\nЭталон:", s["ground_truth"][:300])
print("\nМетрики по сэмплу: rouge1_f={:.4f}, bertscore_f1={:.4f}".format(
    df_gen_baseline.iloc[ex_idx]["rouge1_f"], df_gen_baseline.iloc[ex_idx]["bertscore_f1"]))

Вопрос: Каковы требования к высоте подвески контактного провода над головкой рельса на путях технологического железнодорожного транспорта? ...

Контекст (начало): На время спуска и подъема смены рабочих контактный провод должен отключаться на участке от
ствола до посадочного пункта, расположенного в околоствольном дворе.
1821. На территории промышленной площадки шахты или штольни высота подвески контактного
провода допускается не менее 2,2 м от уровня головки рельса при условии, что откаточные пути не
пересекают проезжих и пешеходных дорог. В местах пересеч ...

Ответ модели: Требования к высоте подвески контактного провода на путях технологического железнодорожного транспорта согласно предоставленному тексту следующие:

*   **На постоянных путях:** высота подвески над головкой рельса должна быть не менее **6250 мм на станциях** и не менее **5750 мм на перегонах** (пункт

Эталон: Высота подвески должна составлять не менее 6250 мм на станционных путях и не менее 5750 мм на перегонах. Да

## 2) Только новый промпт

Меняем только промпт: улучшенная формулировка (роль эксперта, явная инструкция «только из контекста», краткость). Retriever и LLM — те же; retrieval не меняется, сравниваем только generation.

In [9]:
rag_chain_new_prompt = (
    {"context": retriever_baseline | format_docs, "question": RunnablePassthrough()}
    | prompt_improved
    | llm
    | StrOutputParser()
)
answers_new_prompt = [rag_chain_new_prompt.invoke(s["question"]) for s in samples_pdf]
gen_rows_prompt = []
for i, (answer, ref) in enumerate(zip(answers_new_prompt, refs_pdf)):
    r1 = rouge.score(ref, answer)["rouge1"].fmeasure
    rL = rouge.score(ref, answer)["rougeL"].fmeasure
    bleu = sacrebleu.sentence_bleu(answer, [ref], smooth_method="exp").score / 100.0
    gen_rows_prompt.append({"sample_id": i, "rouge1_f": r1, "rougeL_f": rL, "bleu": bleu})
df_gen_new_prompt = pd.DataFrame(gen_rows_prompt)
_, _, F1_p = bert_score_fun(cands=answers_new_prompt, refs=refs_pdf, lang="ru", verbose=False)
df_gen_new_prompt["bertscore_f1"] = F1_p.numpy()
means_gen_new_prompt = df_gen_new_prompt.drop(columns=["sample_id"]).mean()
print("Generation (только новый промпт), средние:")
print(means_gen_new_prompt.to_string())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2033.23it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generation (только новый промпт), средние:
rouge1_f        0.230909
rougeL_f        0.223636
bleu            0.132298
bertscore_f1    0.765783


## 3) Только новая модель эмбеддингов (model_embedding_v2)

Меняем только эмбеддинги: google/gemini-embedding-001 через Open Router. Тот же чанкинг и те же документы; новый векторный store и retriever. Baseline-промпт и baseline-LLM. Сравниваем retrieval и generation.

In [18]:
vectorstore_v2 = Chroma.from_documents(
    documents=splits_pdf,
    embedding=embeddings_v2,
    collection_name="rag_improvements_embedding_v2",
)
retriever_v2 = vectorstore_v2.as_retriever(search_kwargs={"k": 10})
rag_chain_embedding_v2 = (
    {"context": retriever_v2 | format_docs, "question": RunnablePassthrough()}
    | prompt_baseline
    | llm
    | StrOutputParser()
)

retrieval_rows_v2 = []
for i, s in enumerate(samples_pdf):
    retrieved = retriever_v2.invoke(s["question"])
    retrieved_indices = [d.metadata.get("chunk_idx", -1) for d in retrieved]
    rel_idx = s["chunk_idx"]
    rank = 0
    for pos, idx in enumerate(retrieved_indices, 1):
        if idx == rel_idx:
            rank = pos
            break
    recall_at_k = {f"recall@{k}": (1.0 if rel_idx in retrieved_indices[:k] else 0.0) for k in K_LIST}
    mrr = 1.0 / rank if rank else 0.0
    retrieval_rows_v2.append({"sample_id": i, **recall_at_k, "mrr": mrr})
df_ret_v2 = pd.DataFrame(retrieval_rows_v2)
means_ret_v2 = df_ret_v2.drop(columns=["sample_id"]).mean()
print("Retrieval (только embedding v2), средние:")
print(means_ret_v2.to_string())

answers_embedding_v2 = [rag_chain_embedding_v2.invoke(s["question"]) for s in samples_pdf]
gen_rows_v2 = []
for i, (answer, ref) in enumerate(zip(answers_embedding_v2, refs_pdf)):
    r1 = rouge.score(ref, answer)["rouge1"].fmeasure
    rL = rouge.score(ref, answer)["rougeL"].fmeasure
    bleu = sacrebleu.sentence_bleu(answer, [ref], smooth_method="exp").score / 100.0
    gen_rows_v2.append({"sample_id": i, "rouge1_f": r1, "rougeL_f": rL, "bleu": bleu})
df_gen_embedding_v2 = pd.DataFrame(gen_rows_v2)
_, _, F1_v2 = bert_score_fun(cands=answers_embedding_v2, refs=refs_pdf, lang="ru", verbose=False)
df_gen_embedding_v2["bertscore_f1"] = F1_v2.numpy()
means_gen_v2 = df_gen_embedding_v2.drop(columns=["sample_id"]).mean()
print("Generation (только embedding v2), средние:")
print(means_gen_v2.to_string())

Retrieval (только embedding v2), средние:
recall@1     0.560
recall@3     0.840
recall@5     0.960
recall@10    1.000
mrr          0.726


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1683.35it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generation (только embedding v2), средние:
rouge1_f        0.197857
rougeL_f        0.184524
bleu            0.119198
bertscore_f1    0.765032


## 4) Только новая LLM (model_llm_v2)

Меняем только LLM на более мощную модель (model_llm_v2). Тот же retriever и промпт, что в baseline. Retrieval не меняется; сравниваем только generation.

In [19]:
rag_chain_llm_v2 = (
    {"context": retriever_baseline | format_docs, "question": RunnablePassthrough()}
    | prompt_baseline
    | llm_v2
    | StrOutputParser()
)
answers_llm_v2 = [rag_chain_llm_v2.invoke(s["question"]) for s in samples_pdf]
gen_rows_llm_v2 = []
for i, (answer, ref) in enumerate(zip(answers_llm_v2, refs_pdf)):
    r1 = rouge.score(ref, answer)["rouge1"].fmeasure
    rL = rouge.score(ref, answer)["rougeL"].fmeasure
    bleu = sacrebleu.sentence_bleu(answer, [ref], smooth_method="exp").score / 100.0
    gen_rows_llm_v2.append({"sample_id": i, "rouge1_f": r1, "rougeL_f": rL, "bleu": bleu})
df_gen_llm_v2 = pd.DataFrame(gen_rows_llm_v2)
_, _, F1_llm = bert_score_fun(cands=answers_llm_v2, refs=refs_pdf, lang="ru", verbose=False)
df_gen_llm_v2["bertscore_f1"] = F1_llm.numpy()
means_gen_llm_v2 = df_gen_llm_v2.drop(columns=["sample_id"]).mean()
print("Generation (только LLM v2), средние:")
print(means_gen_llm_v2.to_string())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2054.91it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generation (только LLM v2), средние:
rouge1_f        0.184857
rougeL_f        0.182190
bleu            0.073316
bertscore_f1    0.733137


## 5) Все улучшения вместе

Улучшенный промпт + model_embedding_v2 + model_llm_v2. Один векторный store на эмбеддингах v2, retriever k=10, улучшенный промпт и LLM v2.

In [20]:
rag_chain_all = (
    {"context": retriever_v2 | format_docs, "question": RunnablePassthrough()}
    | prompt_improved
    | llm_v2
    | StrOutputParser()
)
answers_all = [rag_chain_all.invoke(s["question"]) for s in samples_pdf]
# Retrieval уже посчитан для retriever_v2 (means_ret_v2, df_ret_v2)
gen_rows_all = []
for i, (answer, ref) in enumerate(zip(answers_all, refs_pdf)):
    r1 = rouge.score(ref, answer)["rouge1"].fmeasure
    rL = rouge.score(ref, answer)["rougeL"].fmeasure
    bleu = sacrebleu.sentence_bleu(answer, [ref], smooth_method="exp").score / 100.0
    gen_rows_all.append({"sample_id": i, "rouge1_f": r1, "rougeL_f": rL, "bleu": bleu})
df_gen_all = pd.DataFrame(gen_rows_all)
_, _, F1_all = bert_score_fun(cands=answers_all, refs=refs_pdf, lang="ru", verbose=False)
df_gen_all["bertscore_f1"] = F1_all.numpy()
means_gen_all = df_gen_all.drop(columns=["sample_id"]).mean()
print("Generation (все улучшения), средние:")
print(means_gen_all.to_string())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1995.01it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generation (все улучшения), средние:
rouge1_f        0.231111
rougeL_f        0.223111
bleu            0.127740
bertscore_f1    0.778988


## 6) Все улучшения + reranker

Добавляем reranker (как в ноутбуке 3): BAAI/bge-reranker-v2-m3, базовый retriever k=15 → после переранжирования в контекст попадают топ-3 документа. Улучшенный промпт и LLM v2; store на эмбеддингах v2.

In [21]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

cross_encoder_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={"device": "cpu"},
)
reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=3)
base_retriever_15 = vectorstore_v2.as_retriever(search_kwargs={"k": 15})
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever_15,
)
rag_chain_all_reranker = (
    {"context": compression_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_improved
    | llm_v2
    | StrOutputParser()
)
print("RAG «все улучшения + reranker» собран (k=15 → top 3).")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1365.57it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]              


RAG «все улучшения + reranker» собран (k=15 → top 3).


In [22]:
retrieval_rows_reranker = []
for i, s in enumerate(samples_pdf):
    retrieved = compression_retriever.invoke(s["question"])
    retrieved_indices = [d.metadata.get("chunk_idx", -1) for d in retrieved]
    rel_idx = s["chunk_idx"]
    rank = 0
    for pos, idx in enumerate(retrieved_indices, 1):
        if idx == rel_idx:
            rank = pos
            break
    recall_at_k = {f"recall@{k}": (1.0 if rel_idx in retrieved_indices[:k] else 0.0) for k in [1, 3, 5, 10]}
    mrr = 1.0 / rank if rank else 0.0
    retrieval_rows_reranker.append({"sample_id": i, **recall_at_k, "mrr": mrr})
df_ret_reranker = pd.DataFrame(retrieval_rows_reranker)
means_ret_reranker = df_ret_reranker.drop(columns=["sample_id"]).mean()
print("Retrieval (все + reranker), средние:")
print(means_ret_reranker.to_string())

answers_all_reranker = [rag_chain_all_reranker.invoke(s["question"]) for s in samples_pdf]
gen_rows_reranker = []
for i, (answer, ref) in enumerate(zip(answers_all_reranker, refs_pdf)):
    r1 = rouge.score(ref, answer)["rouge1"].fmeasure
    rL = rouge.score(ref, answer)["rougeL"].fmeasure
    bleu = sacrebleu.sentence_bleu(answer, [ref], smooth_method="exp").score / 100.0
    gen_rows_reranker.append({"sample_id": i, "rouge1_f": r1, "rougeL_f": rL, "bleu": bleu})
df_gen_reranker = pd.DataFrame(gen_rows_reranker)
_, _, F1_reranker = bert_score_fun(cands=answers_all_reranker, refs=refs_pdf, lang="ru", verbose=False)
df_gen_reranker["bertscore_f1"] = F1_reranker.numpy()
means_gen_reranker = df_gen_reranker.drop(columns=["sample_id"]).mean()
print("Generation (все + reranker), средние:")
print(means_gen_reranker.to_string())

Retrieval (все + reranker), средние:
recall@1     0.800000
recall@3     0.960000
recall@5     0.960000
recall@10    0.960000
mrr          0.873333


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1581.64it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generation (все + reranker), средние:
rouge1_f        0.245333
rougeL_f        0.237333
bleu            0.131602
bertscore_f1    0.772402


## Итоговая таблица сравнения

Сводка средних метрик по всем конфигурациям: baseline, только новый промпт, только новый эмбеддинг, только новая LLM, все улучшения, все + reranker.

In [23]:
metrics_baseline = pd.concat([means_ret_baseline, means_gen_baseline])
metrics_new_prompt = pd.concat([means_ret_baseline, means_gen_new_prompt])  # retrieval = baseline
metrics_embedding_v2 = pd.concat([means_ret_v2, means_gen_v2])
metrics_llm_v2 = pd.concat([means_ret_baseline, means_gen_llm_v2])  # retrieval = baseline
metrics_all = pd.concat([means_ret_v2, means_gen_all])
metrics_all_reranker = pd.concat([means_ret_reranker, means_gen_reranker])

comparison = pd.DataFrame({
    "baseline": metrics_baseline,
    "новый_промпт": metrics_new_prompt,
    "новый_эмбеддинг": metrics_embedding_v2,
    "новая_LLM": metrics_llm_v2,
    "все_улучшения": metrics_all,
    "все_+_reranker": metrics_all_reranker,
})
comparison

,baseline,новый_промпт,новый_эмбеддинг,новая_LLM,все_улучшения,все_+_reranker
recall@1,0.640000,0.640000,0.560000,0.640000,0.560000,0.800000
recall@3,0.880000,0.880000,0.840000,0.880000,0.840000,0.960000
recall@5,0.960000,0.960000,0.960000,0.960000,0.960000,0.960000
recall@10,0.960000,0.960000,1.000000,0.960000,1.000000,0.960000
mrr,0.771333,0.771333,0.726000,0.771333,0.726000,0.873333
rouge1_f,0.205623,0.230909,0.197857,0.184857,0.231111,0.245333
rougeL_f,0.199223,0.223636,0.184524,0.182190,0.223111,0.237333
bleu,0.108106,0.132298,0.119198,0.073316,0.127740,0.131602
bertscore_f1,0.755252,0.765783,0.765032,0.733137,0.778988,0.772402
